<a href="https://colab.research.google.com/github/pantelis/eng-ai-agents/blob/main/notebooks/VLM/clip/clip_zero_shot.ipynb" target="_blank" rel="noopener noreferrer"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Zero-Shot Classification as a Linear Classifier (CLIP)

This notebook demonstrates that zero-shot classification in CLIP can be interpreted as a **linear classifier whose weights are generated from text prompts**.

We will:
1. Derive the formulation
2. Implement zero-shot classification
3. Visualize classifier weights

## 1. Derivation

CLIP learns two embedding functions:

$$
z_I = f_I(I), \quad z_T = f_T(T)
$$

Zero-shot classification uses prompts $T_y$ for each class $y$:

$$
w_y = f_T(T_y)
$$

Prediction:

$$
\hat{y} = \arg\max_y \; z_I^\top w_y
$$

This is exactly a **linear classifier**:

$$
\hat{y} = \arg\max_y \; w_y^\top x
$$

where $x = z_I$.

> Therefore, prompts instantiate classifier weights.

In [ ]:
!pip install transformers torch torchvision pillow matplotlib scikit-learn

In [ ]:
import torch
from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

## 2. Load Image

In [ ]:
url = "https://images.unsplash.com/photo-1518717758536-85ae29035b6d"
image = Image.open(requests.get(url, stream=True).raw)
plt.imshow(image)
plt.axis("off")

## 3. Define Prompts (Classifier Weights)

In [ ]:
labels = ["a dog", "a cat", "a car", "a plane"]
prompts = [f"a photo of {label}" for label in labels]

## 4. Compute Embeddings

In [ ]:
inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds

# Normalize (important for cosine similarity)
image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

## 5. Zero-Shot Classification

$$
\hat{y} = \arg\max_y \; z_I^\top w_y
$$

### Why raw softmax looks flat

CLIP cosine similarities live in a narrow range. Both towers produce L2-normalized embeddings, and their joint space is a tight cone, so raw cosine similarities typically sit between **0.18 and 0.30** even for good matches. A softmax applied directly to such close values produces a nearly uniform distribution, which makes the classifier look much less confident than it actually is.

The fix is the learned **temperature** parameter `logit_scale`. CLIP was trained with a contrastive loss at $\tau \approx 0.01$, equivalently $1/\tau \approx 100$. Inference must scale the similarities by this factor before the softmax:

$$
p(y \mid I) = \mathrm{softmax}_y\left(\frac{1}{\tau} \cdot z_I^\top w_y\right)
$$

HuggingFace's CLIP model exposes this scaled value as `outputs.logits_per_image`, so you get the sharp classification output for free. The cell below compares all three: raw cosine similarity, naive softmax (wrong), temperature-scaled softmax (correct), and the HuggingFace convenience.

In [ ]:
# Raw cosine similarity between image and each text prompt
similarity = (image_embeds @ text_embeds.T).squeeze(0)

# 1. Naive softmax — this is the common pitfall
naive_probs = similarity.softmax(dim=0)

# 2. Correctly scaled softmax using CLIP's learned temperature
#    `logit_scale` is a learned parameter (≈4.6 in log space, ≈100 in linear),
#    and CLIP was trained with contrastive loss at that scale.
logit_scale = model.logit_scale.exp().item()
scaled_probs = (similarity * logit_scale).softmax(dim=0)

# 3. HuggingFace convenience — logits_per_image already has the scale applied
hf_probs = outputs.logits_per_image.softmax(dim=-1).squeeze(0)

print(f"Learned logit_scale (temperature): {logit_scale:.2f}")
print()
print(f"{'label':<10} {'cos sim':>10} {'naive':>10} {'scaled':>10} {'HF':>10}")
print("-" * 55)
for i, label in enumerate(labels):
    print(
        f"{label:<10} "
        f"{similarity[i].item():>10.4f} "
        f"{naive_probs[i].item():>10.4f} "
        f"{scaled_probs[i].item():>10.4f} "
        f"{hf_probs[i].item():>10.4f}"
    )

## 6. Visualizing the classifier weights and the image in the same space

Plotting the first 50 components of each text embedding side by side is not very informative — the embeddings are 512-dimensional and the raw coordinates have no intrinsic meaning. A better view is to project the weights into a low-dimensional subspace and see how they are laid out relative to each other and to the image embedding.

Below we fit a 2-component PCA to the text embeddings and project both the text prompts and the image into the same plane. Each labeled point is one classifier weight $w_y = f_T(T_y)$; the red star is the image embedding $z_I$. The closest label to the star is the zero-shot prediction.

In [ ]:
from sklearn.decomposition import PCA

# Project the text classifier weights to 2D with PCA so we can see them
# arranged in the shared CLIP embedding space. Each point is one prompt,
# and semantically related prompts should cluster together.
weights = text_embeds.numpy()
image_vec = image_embeds.numpy()

pca = PCA(n_components=2)
pca.fit(weights)

proj_text = pca.transform(weights)
proj_image = pca.transform(image_vec)

fig, ax = plt.subplots(figsize=(7, 6))

# Text classifier weights
ax.scatter(proj_text[:, 0], proj_text[:, 1], s=180, c="steelblue", edgecolors="black", zorder=3, label="text prompts (classifier weights)")
for i, label in enumerate(labels):
    ax.annotate(
        label,
        (proj_text[i, 0], proj_text[i, 1]),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=11,
    )

# The image embedding in the same space
ax.scatter(proj_image[:, 0], proj_image[:, 1], s=260, c="crimson", marker="*", edgecolors="black", zorder=4, label="image embedding")

ax.set_xlabel(f"PC1  ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2  ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("CLIP classifier weights and image in 2D (PCA of text embeddings)")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPCA variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
print("Dot-product similarity image · text (closest wins):")
for label, s in zip(labels, (image_vec @ weights.T).flatten()):
    print(f"  {label:<10} {s:.4f}")

## 7. Conclusion

We demonstrated that:

- Prompts generate vectors $w_y$
- These act as classifier weights
- Zero-shot classification is linear classification in embedding space

$$
\hat{y} = \arg\max_y \; w_y^\top x
$$

> Language defines the classifier.